In [1]:
# ============================================
# scripts/setup_and_validate.py
#
# ONE-TIME SETUP SCRIPT — RUN ON KAGGLE ONLY
# Not part of the Streamlit app.
#
# What this does:
#   1. Loads the merged career dataset (O*NET filtered + India-specific)
#   2. Generates sentence embeddings for every career
#   3. Wipes and rebuilds the Pinecone vector index
#   4. Loads seed interaction data and trains the LightSVD model
#   5. Runs a smoke test with cold-start and warm-start profiles
#      to confirm the full recommendation pipeline is healthy
#
# Career data sources (both are used):
#   - onet_careers_filtered.csv : ~800 India-relevant O*NET careers
#   - india_specific_careers.csv: ~60 hand-curated Indian careers
#     (UPSC, CA/CS/CMA, Ayurveda, ISRO, DRDO etc.)
#
# Requires Kaggle Secrets : PINECONE_API_KEY
# Requires Kaggle Dataset : nikhathparveen07/career-system
# ============================================

import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "supabase", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "pinecone", "-q"], check=True)

from kaggle_secrets import UserSecretsClient
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import time
import os
import warnings

warnings.filterwarnings('ignore')

secret_client = UserSecretsClient()

print("✅ All imports successful")
print(f"   numpy  : {np.__version__}")
print(f"   pandas : {pd.__version__}")


# ============================================
# LIGHTSVD MODEL DEFINITION
#
# Biased SVD (Koren et al., 2009):
#   r̂_ui = μ + b_u + b_i + p_u · q_i
#
# Training minimises weighted RMSE with L2 regularisation:
#   L = Σ w_ui (r_ui - r̂_ui)² + λ(||p_u||² + ||q_i||² + b_u² + b_i²)
#
# Where w_ui = 2.0 for real Keep/Drop signals, 1.0 for seed data.
# This is a standalone copy of backend/collaborative.py's LightSVD.
# ============================================

class LightSVD:
    def __init__(self, n_factors=50, n_epochs=20, lr=0.005, reg=0.02):
        self.n_factors   = n_factors
        self.n_epochs    = n_epochs
        self.lr          = lr
        self.reg         = reg
        self.global_mean = 3.5

    def fit(self, ratings_df):
        """
        Train on DataFrame with columns: student_id, career_id, rating, weight.
        weight=2.0 for real interactions, weight=1.0 for seed data.
        """
        users  = ratings_df['student_id'].unique()
        items  = ratings_df['career_id'].unique()

        self.user_index  = {u: i for i, u in enumerate(users)}
        self.item_index  = {it: i for i, it in enumerate(items)}
        self.global_mean = ratings_df['rating'].mean()

        n_users = len(users)
        n_items = len(items)

        np.random.seed(42)
        self.P  = np.random.normal(0, 0.1, (n_users, self.n_factors))
        self.Q  = np.random.normal(0, 0.1, (n_items, self.n_factors))
        self.bu = np.zeros(n_users)
        self.bi = np.zeros(n_items)

        for epoch in range(self.n_epochs):
            for _, row in ratings_df.iterrows():
                u = self.user_index.get(row['student_id'])
                i = self.item_index.get(row['career_id'])
                if u is None or i is None:
                    continue
                weight     = row.get('weight', 1.0)
                err        = row['rating'] - self._raw(u, i)
                self.bu[u] += self.lr * (weight * err - self.reg * self.bu[u])
                self.bi[i] += self.lr * (weight * err - self.reg * self.bi[i])
                pu         = self.P[u].copy()
                self.P[u] += self.lr * (weight * err * self.Q[i] - self.reg * self.P[u])
                self.Q[i] += self.lr * (weight * err * pu        - self.reg * self.Q[i])

        print(f"✅ LightSVD trained — {n_users} users, {n_items} items, "
              f"global mean: {self.global_mean:.2f}")
        return self

    def _raw(self, u_idx, i_idx):
        return (self.global_mean +
                self.bu[u_idx] +
                self.bi[i_idx] +
                np.dot(self.P[u_idx], self.Q[i_idx]))

    def predict(self, user_id, career_id):
        u = self.user_index.get(user_id)
        i = self.item_index.get(career_id)
        if u is None or i is None:
            return self.global_mean
        return float(np.clip(self._raw(u, i), 1, 5))


# ============================================
# ADAPTIVE WEIGHT SCHEDULE
# Mirrors backend/hybrid_engine.py exactly.
# Four stages based on real interaction count per student.
# ============================================

WEIGHT_STAGES = [
    (1,  1.00, 0.00),   # Stage 0: cold start — content only (n = 0)
    (10, 0.85, 0.15),   # Stage 1: sparse
    (30, 0.70, 0.30),   # Stage 2: moderate
    (float('inf'), 0.55, 0.45),  # Stage 3: rich
]

def get_adaptive_weights(n_real_interactions):
    for threshold, cw, sw in WEIGHT_STAGES:
        if n_real_interactions < threshold:
            return cw, sw
    return 0.55, 0.45


# ============================================
# STEP 1 — LOAD BOTH CAREER DATASETS
#
# We load and merge two sources:
#   1. onet_careers_filtered.csv — O*NET careers after India relevance
#      filter (removes ~200 inapplicable roles like military, gambling,
#      postal service etc.). ~800 careers remain.
#   2. india_specific_careers.csv — ~60 careers O*NET does not cover:
#      UPSC civil services, CA/CS/CMA professional qualifications,
#      AYUSH practitioners, ISRO/DRDO scientists, state PSC officers,
#      banking exams (IBPS/RBI/SEBI), etc.
#
# Both files share the same column schema. drop_duplicates on job_title
# keeps the O*NET entry if both sources have the same title.
# ============================================

print("\n📂 Loading career datasets...")

BASE = "/kaggle/input/datasets/nikhathparveen07/career-system"

# ── Load O*NET filtered careers ───────────────────────────────
onet_path = f"{BASE}/onet_careers_filtered.csv"
if not os.path.exists(onet_path):
    raise FileNotFoundError(f"O*NET file not found at {onet_path}")

onet_df = pd.read_csv(onet_path)
print(f"   O*NET filtered: {len(onet_df)} careers")

# Normalise column names
if "12th_stream" in onet_df.columns:
    onet_df.rename(columns={"12th_stream": "stream"}, inplace=True)
if "onet_code" in onet_df.columns and "nco_id" not in onet_df.columns:
    onet_df.rename(columns={"onet_code": "nco_id"}, inplace=True)

# ── Load India-specific careers ───────────────────────────────
india_path = f"{BASE}/india_specific_careers.csv"
if not os.path.exists(india_path):
    raise FileNotFoundError(f"India-specific file not found at {india_path}")

india_df = pd.read_csv(india_path)
print(f"   India-specific: {len(india_df)} careers")

if "12th_stream" in india_df.columns:
    india_df.rename(columns={"12th_stream": "stream"}, inplace=True)

# ── Merge on shared columns only ─────────────────────────────
shared_cols = [c for c in onet_df.columns if c in india_df.columns]
print(f"   Shared columns: {shared_cols}")

india_df_aligned = india_df[shared_cols]
onet_df_aligned  = onet_df[shared_cols]

df = pd.concat([onet_df_aligned, india_df_aligned], ignore_index=True).drop_duplicates(
    subset="job_title", keep="first"
).reset_index(drop=True)

print(f"\n✅ Total careers after merge: {len(df)}")
print(f"\n🎯 Stream distribution:")
print(df['stream'].value_counts())
print(f"\n🏭 Top sectors:")
print(df['sector'].value_counts().head(10))

# Sanity check — India-specific entries present
india_titles = india_df['job_title'].tolist()
india_found  = df[df['job_title'].isin(india_titles)]
print(f"\n🇮🇳 India-specific entries in merged dataset: {len(india_found)}")
assert len(india_found) > 50, "ERROR: India-specific careers missing from merged dataset"
print("✅ India-specific career merge confirmed")


# ============================================
# STEP 2 — GENERATE SENTENCE EMBEDDINGS
#
# Model: all-MiniLM-L6-v2 (384-dim)
# Same model used in production (backend/embeddings.py).
# Text format: "job_title. sector. RIASEC: X Y. Skills: ..."
# ============================================

print("\n🔄 Loading Sentence Transformer model (all-MiniLM-L6-v2)...")
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded")

if os.path.exists('career_embeddings.npy'):
    os.remove('career_embeddings.npy')

df['text_for_embedding'] = (
    df['job_title'].fillna('') + '. ' +
    df['sector'].fillna('') + '. ' +
    'RIASEC: ' + df['primary_riasec'].fillna('') + ' ' +
    df['secondary_riasec'].fillna('') + '. ' +
    'Skills: ' + df['core_skills'].fillna('')
)

print(f"\n🔄 Generating embeddings for {len(df)} careers...")
embeddings = sentence_model.encode(
    df['text_for_embedding'].tolist(),
    show_progress_bar=True,
    batch_size=64
)

np.save('career_embeddings.npy', embeddings)
print(f"✅ Embeddings generated — shape: {embeddings.shape}")

# Spot check: embedding for an India-specific career
india_row_idx = df[df['job_title'] == 'IAS Officer (Indian Administrative Service)'].index
if len(india_row_idx) > 0:
    idx = india_row_idx[0]
    print(f"✅ Spot check — IAS Officer embedding dim: {embeddings[idx].shape}, norm: {np.linalg.norm(embeddings[idx]):.3f}")
else:
    print("⚠️  IAS Officer not found — check India-specific file")


# ============================================
# STEP 3 — REBUILD PINECONE INDEX
#
# Full wipe + rebuild is safer than incremental upsert because:
#   - Deleted careers from old O*NET fetches are fully removed
#   - Vector IDs (career_0, career_1...) stay in sync with df.iloc
#   - No stale vectors bleed into recommendations
# ============================================

print("\n🔄 Connecting to Pinecone...")
pc         = Pinecone(api_key=secret_client.get_secret("PINECONE_API_KEY"))
index_name = "career-discovery"

existing = [i.name for i in pc.list_indexes()]
if index_name in existing:
    print("🗑️  Deleting old index...")
    pc.delete_index(index_name)
    time.sleep(10)

print("🔄 Creating fresh index (dim=384, metric=cosine)...")
pc.create_index(
    name      = index_name,
    dimension = 384,
    metric    = "cosine",
    spec      = ServerlessSpec(cloud="aws", region="us-east-1")
)
time.sleep(15)
print("✅ Index created")

index = pc.Index(index_name)

# Build vectors — metadata must contain all fields read by app.py
vectors = []
for i, row in df.iterrows():
    vectors.append({
        "id":     f"career_{i}",
        "values": embeddings[i].tolist(),
        "metadata": {
            "job_title":        str(row.get('job_title', '')),
            "sector":           str(row.get('sector', '')),
            "stream":           str(row.get('stream', '')),
            "nco_id":           str(row.get('nco_id', '')),
            "primary_riasec":   str(row.get('primary_riasec', '')),
            "secondary_riasec": str(row.get('secondary_riasec', ''))
        }
    })

print(f"\n🔄 Uploading {len(vectors)} vectors to Pinecone...")
for i in range(0, len(vectors), 100):
    index.upsert(vectors=vectors[i:i+100])
    if (i // 100 + 1) % 5 == 0:
        print(f"   ✅ Uploaded {min(i+100, len(vectors))}/{len(vectors)}")

time.sleep(5)
stats = index.describe_index_stats()
print(f"\n✅ Pinecone index ready — total vectors: {stats['total_vector_count']}")

assert stats['total_vector_count'] == len(df), (
    f"MISMATCH: {stats['total_vector_count']} vectors vs {len(df)} careers"
)
print("✅ Vector count matches career count")


# ============================================
# STEP 4 — TRAIN LIGHTSVD ON SEED INTERACTIONS
#
# Seed data: 100 synthetic students × ~10 interactions each.
# Stream-stratified: each student rates careers from their own stream
# and a small number of cross-stream careers with low ratings.
# Weight = 1.0 for seed (vs 2.0 for real Keep/Drop in production).
# ============================================

print("\n📂 Loading seed interactions...")

interactions_path = f"{BASE}/interactions_final.csv"
if not os.path.exists(interactions_path):
    # Try alternate filename (spaces)
    alt_path = f"{BASE}/interactions (1).csv"
    if os.path.exists(alt_path):
        interactions_path = alt_path
    else:
        raise FileNotFoundError(f"interactions.csv not found at {BASE}")

interactions_df = pd.read_csv(interactions_path)
interactions_df = interactions_df[['student_id', 'career_id', 'rating']].copy()
interactions_df['weight'] = 1.0
interactions_df['source'] = 'seed'

print(f"✅ Loaded {len(interactions_df)} seed interactions")
print(f"   Unique students : {interactions_df['student_id'].nunique()}")
print(f"   Unique careers  : {interactions_df['career_id'].nunique()}")
print(f"   Rating dist     :\n{interactions_df['rating'].value_counts().sort_index()}")

print("\n🔄 Training LightSVD (n_factors=50, n_epochs=20, lr=0.005, λ=0.02)...")
svd_model = LightSVD(n_factors=50, n_epochs=20, lr=0.005, reg=0.02)
svd_model.fit(interactions_df)


# ============================================
# HELPER — RIASEC BOOST
# Mirrors backend/content_filter.py exactly.
# ============================================

def get_riasec_boost(career_row, student_riasec):
    primary   = career_row.get('primary_riasec', '')
    secondary = career_row.get('secondary_riasec', '')
    top_type  = student_riasec['top2'][0]
    sec_type  = student_riasec['top2'][1]

    if primary == top_type or secondary == top_type:
        return 1.3
    elif primary == sec_type or secondary == sec_type:
        return 1.15
    return 1.0


# ============================================
# HELPER — HYBRID RECOMMENDATION ENGINE
# Mirrors backend/hybrid_engine.py exactly.
# Cold-start logic uses get_adaptive_weights().
# ============================================

def search_by_text(query_text, top_k=20):
    query_embedding = sentence_model.encode(query_text).tolist()
    return index.query(vector=query_embedding, top_k=top_k, include_metadata=True)


def hybrid_recommendations(
        user_id, query_text, student_riasec,
        n_real_interactions=0,
        preferred_stream=None,
        top_k=10):
    """
    Full hybrid recommendation pipeline.

    Cold-start: when n_real_interactions == 0, operates as pure content
    retrieval. The collaborative component activates progressively
    as n_real_interactions grows through the four stages.
    """
    # Stage-based adaptive weights
    is_cold_start = (n_real_interactions == 0)
    content_weight, collab_weight = get_adaptive_weights(n_real_interactions)

    # Content retrieval
    results_raw = search_by_text(query_text, top_k=20)

    # Collaborative scores (skipped in cold start)
    collab_scores = {}
    if not is_cold_start and collab_weight > 0:
        raw = {i: svd_model.predict(user_id, i) for i in range(len(df))}
        mn  = min(raw.values())
        mx  = max(raw.values())
        collab_scores = {
            i: (raw[i] - mn) / (mx - mn) if mx > mn else 0.5
            for i in raw
        }

    hybrid_scores = []
    for match in results_raw['matches']:
        try:
            career_id = int(match['id'].split('_')[1])
        except (ValueError, IndexError):
            continue

        if career_id >= len(df):
            continue

        career_row    = df.iloc[career_id]
        career_stream = match['metadata'].get('stream', '')
        content_score = match['score']

        # Collaborative score with cross-stream penalty
        collab_score = collab_scores.get(career_id, 0.0)
        if not is_cold_start and preferred_stream and career_stream != preferred_stream:
            collab_score *= 0.5  # cross-stream penalty

        # Post-score multipliers
        stream_boost = 1.2 if (preferred_stream and career_stream == preferred_stream) else 1.0
        riasec_boost = get_riasec_boost(career_row.to_dict(), student_riasec)

        base_score  = content_weight * content_score + collab_weight * collab_score
        final_score = base_score * stream_boost * riasec_boost

        hybrid_scores.append({
            'career_id':           career_id,
            'career':              career_row['job_title'],
            'sector':              career_row.get('sector', ''),
            'stream':              career_stream,
            'primary_riasec':      career_row.get('primary_riasec', ''),
            'secondary_riasec':    career_row.get('secondary_riasec', ''),
            'final_score':         round(final_score,   4),
            'content_score':       round(content_score, 4),
            'collab_score':        round(collab_score,  4),
            'stream_boost':        stream_boost,
            'riasec_boost':        riasec_boost,
            'is_cold_start':       is_cold_start,
            'content_weight':      content_weight,
            'collab_weight':       collab_weight,
            'n_real_interactions': n_real_interactions,
        })

    hybrid_scores.sort(key=lambda x: x['final_score'], reverse=True)
    return hybrid_scores[:top_k]


# ============================================
# STEP 5 — VALIDATION
#
# Tests 6 student profiles:
#   - 4 cold-start (n_real=0): pure content mode
#   - 2 warm-start (n_real=15): hybrid mode with collab weight
#
# For each profile, checks:
#   a) Results are returned
#   b) At least 3/5 results match the student's stream
#   c) Cold-start profiles use content_weight=1.0
#   d) Warm-start profiles use content_weight < 1.0
#   e) India-specific careers appear in results where expected
# ============================================

print("\n" + "="*65)
print("VALIDATION: Testing 6 student profiles (4 cold + 2 warm)")
print("="*65)

test_cases = [
    # Cold-start profiles (n_real=0 → content only)
    {
        "name": "Riya", "user_id": "STU_TEST_0001", "stream": "Science",
        "query": "I love biology working in labs discovering medicines",
        "riasec": {'top2': ['I', 'R'], 'riasec_code': 'IR'},
        "n_real": 0, "mode": "cold-start"
    },
    {
        "name": "Arjun", "user_id": "STU_TEST_0002", "stream": "Commerce",
        "query": "I enjoy working with money stocks financial planning",
        "riasec": {'top2': ['C', 'E'], 'riasec_code': 'CE'},
        "n_real": 0, "mode": "cold-start"
    },
    {
        "name": "Priya", "user_id": "STU_TEST_0003", "stream": "Arts",
        "query": "I love drawing creating designs visual storytelling",
        "riasec": {'top2': ['A', 'S'], 'riasec_code': 'AS'},
        "n_real": 0, "mode": "cold-start"
    },
    {
        "name": "Rahul", "user_id": "STU_TEST_0004", "stream": "Vocational",
        "query": "I enjoy fixing machines electrical systems hands on repair",
        "riasec": {'top2': ['R', 'C'], 'riasec_code': 'RC'},
        "n_real": 0, "mode": "cold-start"
    },
    # Warm-start profiles (n_real=15 → Stage 1: 85/15 split)
    {
        "name": "Ananya", "user_id": "STU1026", "stream": "Science",
        "query": "biotech medical laboratory research experiments",
        "riasec": {'top2': ['I', 'R'], 'riasec_code': 'IR'},
        "n_real": 15, "mode": "warm-start (Stage 1)"
    },
    {
        "name": "Gaurav", "user_id": "STU1063", "stream": "Commerce",
        "query": "banking investment stock market trading financial analysis",
        "riasec": {'top2': ['E', 'S'], 'riasec_code': 'ES'},
        "n_real": 15, "mode": "warm-start (Stage 1)"
    },
]

all_passed = True

for t in test_cases:
    print(f"\n{'='*65}")
    print(f"👤 {t['name']} ({t['mode']}) — {t['stream']} / {t['riasec']['riasec_code']}")
    print(f"   n_real={t['n_real']} → expected weights: {get_adaptive_weights(t['n_real'])}")
    print(f"💭 {t['query']}")
    print(f"{'='*65}")

    results = hybrid_recommendations(
        user_id              = t['user_id'],
        query_text           = t['query'],
        student_riasec       = t['riasec'],
        n_real_interactions  = t['n_real'],
        preferred_stream     = t['stream'],
        top_k                = 5,
    )

    if not results:
        print(f"  ❌ FAIL — No results returned")
        all_passed = False
        continue

    first = results[0]
    actual_cw = first['content_weight']
    actual_sw = first['collab_weight']
    expected_cw, expected_sw = get_adaptive_weights(t['n_real'])

    # Check weights match expected stage
    if abs(actual_cw - expected_cw) > 0.001:
        print(f"  ❌ WEIGHT MISMATCH — got ({actual_cw}, {actual_sw}), expected ({expected_cw}, {expected_sw})")
        all_passed = False
    else:
        print(f"  ✅ Weights correct — content: {actual_cw}, collab: {actual_sw}")

    # Check cold-start flag
    if t['n_real'] == 0 and not first['is_cold_start']:
        print(f"  ❌ is_cold_start should be True for n_real=0")
        all_passed = False
    if t['n_real'] > 0 and first['is_cold_start']:
        print(f"  ❌ is_cold_start should be False for n_real={t['n_real']}")
        all_passed = False

    # Check stream match count
    stream_matches = sum(1 for r in results if r['stream'] == t['stream'])

    for i, r in enumerate(results):
        stream_icon = "🎓" if r['stream_boost'] > 1.0 else "  "
        riasec_icon = "⭐⭐" if r['riasec_boost'] == 1.3 else ("⭐" if r['riasec_boost'] == 1.15 else "  ")
        cold_label  = "[CONTENT]" if r['is_cold_start'] else "[HYBRID]"
        print(f"  {i+1}. {stream_icon}{riasec_icon} {cold_label} {r['career']} "
              f"({r['stream']}) — score: {r['final_score']}")

    if stream_matches < 3:
        print(f"  ⚠️  Only {stream_matches}/5 from {t['stream']} stream")
        all_passed = False
    else:
        print(f"  ✅ {stream_matches}/5 from {t['stream']} stream")

# ── Check India-specific careers appear in results ────────────
print(f"\n{'='*65}")
print("INDIA-SPECIFIC CAREER SPOT CHECKS")
print(f"{'='*65}")

india_queries = [
    {
        "query": "government civil services public administration India UPSC",
        "stream": "Arts",
        "riasec": {'top2': ['E', 'S'], 'riasec_code': 'ES'},
        "expected_keywords": ["IAS", "IPS", "IFS", "SDM", "PSC"],
    },
    {
        "query": "chartered accountant audit taxation corporate finance India",
        "stream": "Commerce",
        "riasec": {'top2': ['C', 'E'], 'riasec_code': 'CE'},
        "expected_keywords": ["Chartered Accountant", "CA", "CMA", "Company Secretary"],
    },
    {
        "query": "ISRO space research defence scientist engineering India",
        "stream": "Science",
        "riasec": {'top2': ['I', 'R'], 'riasec_code': 'IR'},
        "expected_keywords": ["ISRO", "DRDO", "PSU", "GATE"],
    },
]

for q in india_queries:
    results = hybrid_recommendations(
        user_id="SPOT_CHECK",
        query_text=q['query'],
        student_riasec=q['riasec'],
        n_real_interactions=0,
        preferred_stream=q['stream'],
        top_k=10,
    )
    titles = [r['career'] for r in results]
    found  = [k for k in q['expected_keywords'] if any(k in t for t in titles)]
    if found:
        print(f"  ✅ India-specific found for '{q['query'][:40]}...' : {found[:3]}")
    else:
        print(f"  ⚠️  No India-specific match for '{q['query'][:40]}...'")
        print(f"     Top results: {titles[:5]}")
        # Not marking as failure — depends on embedding similarity which may vary

print(f"\n{'='*65}")
if all_passed:
    print("✅ ALL VALIDATION CHECKS PASSED — pipeline is healthy")
    print("   Both O*NET and India-specific careers are indexed.")
    print("   Cold-start and warm-start modes work correctly.")
else:
    print("⚠️  SOME CHECKS FAILED — review warnings above before deploying")
print("="*65)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.7 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 6.0 MB/s eta 0:00:00
✅ All imports successful
   numpy  : 2.0.2
   pandas : 2.3.3

📂 Loading career datasets...
   O*NET filtered: 850 careers
   India-specific: 55 careers
   Shared columns: ['nco_id', 'job_title', 'sector', 'stream', 'primary_riasec', 'secondary_riasec', 'core_skills', 'source', 'min_education']

✅ Total careers after merge: 905

🎯 Stream distribution:
stream
Vocational    303
Science       291
Commerce      180
Arts          131
Name: count, dtype: int64

🏭 Top sectors:
sector
Healthcare              113
Manufacturing           106
Education                63
Research & Science       60
Construction             59
Engineering              57
Management               54
Maintenance & Repair     52
Finance                  49
Administration           46
Name: count, dtype: int64

🇮🇳 India-specific entries in merged dataset: 55
✅ India

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

... (9 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded

🔄 Generating embeddings for 905 careers...


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

✅ Embeddings generated — shape: (905, 384)
✅ Spot check — IAS Officer embedding dim: (384,), norm: 1.000

🔄 Connecting to Pinecone...
🗑️  Deleting old index...
🔄 Creating fresh index (dim=384, metric=cosine)...
✅ Index created

🔄 Uploading 905 vectors to Pinecone...
   ✅ Uploaded 500/905
   ✅ Uploaded 905/905

✅ Pinecone index ready — total vectors: 905
✅ Vector count matches career count

📂 Loading seed interactions...
✅ Loaded 1409 seed interactions
   Unique students : 100
   Unique careers  : 707
   Rating dist     :
rating
1    135
2    140
3    215
4    380
5    539
Name: count, dtype: int64

🔄 Training LightSVD (n_factors=50, n_epochs=20, lr=0.005, λ=0.02)...
✅ LightSVD trained — 100 users, 707 items, global mean: 3.74

VALIDATION: Testing 6 student profiles (4 cold + 2 warm)

👤 Riya (cold-start) — Science / IR
   n_real=0 → expected weights: (1.0, 0.0)
💭 I love biology working in labs discovering medicines
  ✅ Weights correct — content: 1.0, collab: 0.0
  1. 🎓⭐⭐ [CONTENT] Bioch

# **Evaluation**

In [2]:
# ============================================================
# evaluation_pipeline.py
#
# Standalone evaluation script for the Hybrid Career
# Recommendation System — Stages 0 through 3 + Ablation.
#
# Designed to run on Kaggle with access to:
#   - onet_careers_filtered.csv
#   - india_specific_careers.csv
#   - Pinecone index (career-discovery)
#
# What this script does:
#   1. Builds 50 synthetic student profiles
#   2. Runs Stage 0 evaluation (pure content)
#   3. Simulates Keep/Drop signals for Stages 1-3
#   4. Runs Stage 1, 2, 3 evaluation (hybrid)
#   5. Runs ablation study (Stage 0 only, 5 variants)
#   6. Computes all metrics and prints summary tables
#   7. Saves results to eval_results.csv
#
# Metrics computed:
#   - RIASEC Alignment @5
#   - Stream Relevance @5
#   - Precision@5 (ground truth = correct stream + RIASEC)
#   - NDCG@5
#   - Career Diversity (unique titles)
#   - Catalog Coverage
#   - Top-5 Jaccard distance from Stage 0 (Stages 1-3)
#   - Mean SVD score contribution (Stages 1-3)
# ============================================================

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "pinecone", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "sentence-transformers", "-q"], check=True)

import pandas as pd
import numpy as np
import math
import os
import time
import warnings
from collections import defaultdict

warnings.filterwarnings('ignore')

# ── Pinecone + SentenceTransformer ───────────────────────────
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer

# ============================================================
# CONFIGURATION — edit paths if needed
# ============================================================

BASE          = "/kaggle/input/datasets/nikhathparveen07/career-system"
ONET_PATH     = f"{BASE}/onet_careers_filtered.csv"
INDIA_PATH    = f"{BASE}/india_specific_careers.csv"
PINECONE_KEY  = None   # filled from Kaggle secrets below
INDEX_NAME    = "career-discovery"
TOP_K         = 5
N_PROFILES    = 50

# ============================================================
# LOAD SECRETS
# ============================================================
try:
    from kaggle_secrets import UserSecretsClient
    PINECONE_KEY = UserSecretsClient().get_secret("PINECONE_API_KEY")
    print("✅ Pinecone API key loaded from Kaggle secrets")
except Exception:
    PINECONE_KEY = os.environ.get("PINECONE_API_KEY", "")
    print("⚠️  Kaggle secrets unavailable — reading from env PINECONE_API_KEY")

# ============================================================
# LOAD CAREER DATABASE
# ============================================================
print("\n📂 Loading career database...")

onet_df  = pd.read_csv(ONET_PATH)
india_df = pd.read_csv(INDIA_PATH)

for frame in [onet_df, india_df]:
    if "12th_stream" in frame.columns:
        frame.rename(columns={"12th_stream": "stream"}, inplace=True)
    if "onet_code" in frame.columns and "nco_id" not in frame.columns:
        frame.rename(columns={"onet_code": "nco_id"}, inplace=True)

shared_cols = [c for c in onet_df.columns if c in india_df.columns]
df = pd.concat(
    [onet_df[shared_cols], india_df[shared_cols]], ignore_index=True
).drop_duplicates(subset="job_title", keep="first").reset_index(drop=True)

print(f"✅ Total careers: {len(df)}")
print(df['stream'].value_counts().to_string())

# ============================================================
# LIGHTSVD — same as production
# ============================================================

class LightSVD:
    def __init__(self, n_factors=50, n_epochs=20, lr=0.005, reg=0.02):
        self.n_factors   = n_factors
        self.n_epochs    = n_epochs
        self.lr          = lr
        self.reg         = reg
        self.global_mean = 3.5

    def fit(self, ratings_df):
        users  = ratings_df['student_id'].unique()
        items  = ratings_df['career_id'].unique()
        self.user_index  = {u: i for i, u in enumerate(users)}
        self.item_index  = {it: i for i, it in enumerate(items)}
        self.global_mean = ratings_df['rating'].mean()
        n_users = len(users)
        n_items = len(items)
        np.random.seed(42)
        self.P  = np.random.normal(0, 0.1, (n_users, self.n_factors))
        self.Q  = np.random.normal(0, 0.1, (n_items, self.n_factors))
        self.bu = np.zeros(n_users)
        self.bi = np.zeros(n_items)
        for _ in range(self.n_epochs):
            for _, row in ratings_df.iterrows():
                u = self.user_index.get(row['student_id'])
                i = self.item_index.get(row['career_id'])
                if u is None or i is None:
                    continue
                weight     = row.get('weight', 1.0)
                err        = row['rating'] - self._raw(u, i)
                self.bu[u] += self.lr * (weight * err - self.reg * self.bu[u])
                self.bi[i] += self.lr * (weight * err - self.reg * self.bi[i])
                pu         = self.P[u].copy()
                self.P[u] += self.lr * (weight * err * self.Q[i] - self.reg * self.P[u])
                self.Q[i] += self.lr * (weight * err * pu        - self.reg * self.Q[i])
        return self

    def _raw(self, u, i):
        return (self.global_mean + self.bu[u] + self.bi[i] +
                np.dot(self.P[u], self.Q[i]))

    def predict(self, user_id, career_id):
        u = self.user_index.get(user_id)
        i = self.item_index.get(career_id)
        if u is None or i is None:
            return self.global_mean
        return float(np.clip(self._raw(u, i), 1, 5))


# ============================================================
# ADAPTIVE WEIGHTS — same as production
# ============================================================

WEIGHT_STAGES = [
    (1,          1.00, 0.00),
    (10,         0.85, 0.15),
    (30,         0.70, 0.30),
    (float('inf'), 0.55, 0.45),
]

def get_adaptive_weights(n_real):
    for threshold, cw, sw in WEIGHT_STAGES:
        if n_real < threshold:
            return cw, sw
    return 0.55, 0.45


# ============================================================
# RIASEC BOOST — same as production
# ============================================================

def get_riasec_boost(career_row, student_riasec):
    primary   = career_row.get('primary_riasec', '')
    secondary = career_row.get('secondary_riasec', '')
    top_type  = student_riasec['top2'][0]
    sec_type  = student_riasec['top2'][1]
    if primary == top_type or secondary == top_type:
        return 1.3
    elif primary == sec_type or secondary == sec_type:
        return 1.15
    return 1.0


# ============================================================
# SENTENCE TRANSFORMER + PINECONE
# ============================================================

print("\n🔄 Loading Sentence Transformer...")
sbert = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ SBERT loaded")

print("🔄 Connecting to Pinecone...")
pc    = Pinecone(api_key=PINECONE_KEY)
index = pc.Index(INDEX_NAME)
print("✅ Pinecone connected")


def search_pinecone(query_text, top_k=20):
    vec = sbert.encode(query_text).tolist()
    return index.query(vector=vec, top_k=top_k, include_metadata=True)


# ============================================================
# HYBRID RECOMMENDATION ENGINE
# Supports 5 ablation modes:
#   "full"         — full system (default)
#   "no_llm"       — no LLM expansion (raw query, handled outside)
#   "no_riasec"    — B_RIASEC = 1.0
#   "no_stream"    — B_stream = 1.0, no cross-stream penalty
#   "baseline_b"   — pure RIASEC lookup, no semantic retrieval
# ============================================================

def recommend(
        user_id, query_text, student_riasec,
        n_real_interactions=0,
        preferred_stream=None,
        svd_model=None,
        mode="full",
        top_k=TOP_K):

    # ── Baseline B: pure RIASEC lookup, no semantic matching ─
    if mode == "baseline_b":
        primary_type = student_riasec['top2'][0]
        matches = df[df['primary_riasec'] == primary_type].head(top_k)
        return [
            {
                'career_id':        idx,
                'career':           row['job_title'],
                'stream':           row.get('stream', ''),
                'primary_riasec':   row.get('primary_riasec', ''),
                'secondary_riasec': row.get('secondary_riasec', ''),
                'final_score':      1.0,
                'content_score':    0.0,
                'collab_score':     0.0,
                'content_weight':   0.0,
                'collab_weight':    0.0,
            }
            for idx, row in matches.iterrows()
        ]

    is_cold_start = (n_real_interactions == 0)
    content_weight, collab_weight = get_adaptive_weights(n_real_interactions)

    # Content retrieval
    results_raw = search_pinecone(query_text, top_k=20)

    # Collaborative scores
    collab_scores = {}
    if not is_cold_start and collab_weight > 0 and svd_model is not None:
        raw = {i: svd_model.predict(user_id, i) for i in range(len(df))}
        mn, mx = min(raw.values()), max(raw.values())
        collab_scores = {
            i: (raw[i] - mn) / (mx - mn) if mx > mn else 0.5
            for i in raw
        }

    hybrid_scores = []
    for match in results_raw['matches']:
        try:
            career_id = int(match['id'].split('_')[1])
        except (ValueError, IndexError):
            continue
        if career_id >= len(df):
            continue

        career_row    = df.iloc[career_id]
        career_stream = match['metadata'].get('stream', '')
        content_score = match['score']

        collab_score = collab_scores.get(career_id, 0.0)
        if not is_cold_start and preferred_stream and career_stream != preferred_stream:
            collab_score *= 0.85  # cross-stream penalty

        # Multipliers — ablation controls
        stream_boost = 1.0
        if mode != "no_stream":
            stream_boost = 1.2 if (preferred_stream and career_stream == preferred_stream) else 1.0

        riasec_boost = 1.0
        if mode != "no_riasec":
            riasec_boost = get_riasec_boost(career_row.to_dict(), student_riasec)

        base_score  = content_weight * content_score + collab_weight * collab_score
        final_score = base_score * stream_boost * riasec_boost

        hybrid_scores.append({
            'career_id':        career_id,
            'career':           career_row['job_title'],
            'stream':           career_stream,
            'primary_riasec':   career_row.get('primary_riasec', ''),
            'secondary_riasec': career_row.get('secondary_riasec', ''),
            'final_score':      round(final_score, 4),
            'content_score':    round(content_score, 4),
            'collab_score':     round(collab_score, 4),
            'content_weight':   content_weight,
            'collab_weight':    collab_weight,
        })

    hybrid_scores.sort(key=lambda x: x['final_score'], reverse=True)
    return hybrid_scores[:top_k]


# ============================================================
# METRIC FUNCTIONS
# ============================================================

def is_relevant(career, profile):
    """
    Ground truth relevance:
    A career is relevant if it matches the student's stream
    AND its primary or secondary RIASEC matches the student's
    primary OR secondary type.
    Stated clearly in paper: synthetic ground truth definition.
    """
    stream_match = (career['stream'] == profile['stream'])
    p_riasec     = career.get('primary_riasec', '')
    s_riasec     = career.get('secondary_riasec', '')
    student_p    = profile['riasec']['top2'][0]
    student_s    = profile['riasec']['top2'][1]
    riasec_match = (p_riasec in [student_p, student_s] or
                    s_riasec in [student_p, student_s])
    return stream_match and riasec_match


def precision_at_k(results, profile, k=TOP_K):
    relevant = sum(1 for r in results[:k] if is_relevant(r, profile))
    return relevant / k


def ndcg_at_k(results, profile, k=TOP_K):
    """NDCG@K with binary relevance."""
    dcg  = sum(
        int(is_relevant(r, profile)) / math.log2(i + 2)
        for i, r in enumerate(results[:k])
    )
    # Ideal DCG: all relevant items at the top
    n_relevant = sum(1 for r in results[:k] if is_relevant(r, profile))
    idcg = sum(1 / math.log2(i + 2) for i in range(min(n_relevant, k)))
    return dcg / idcg if idcg > 0 else 0.0


def riasec_alignment(results, profile, k=TOP_K):
    """% of top-k results whose primary_riasec matches student's primary."""
    primary = profile['riasec']['top2'][0]
    matches = sum(
        1 for r in results[:k]
        if r.get('primary_riasec', '') == primary or
           r.get('secondary_riasec', '') == primary
    )
    return matches / k


def stream_relevance(results, profile, k=TOP_K):
    """% of top-k results matching student's stream."""
    matches = sum(1 for r in results[:k] if r['stream'] == profile['stream'])
    return matches / k


def jaccard_distance(list_a, list_b):
    """1 - Jaccard similarity between two top-k career title sets."""
    set_a = set(r['career'] for r in list_a)
    set_b = set(r['career'] for r in list_b)
    if not set_a and not set_b:
        return 0.0
    intersection = len(set_a & set_b)
    union        = len(set_a | set_b)
    return 1.0 - intersection / union


def mean_svd_contribution(results):
    """Average collab_weight * collab_score across results."""
    if not results:
        return 0.0
    return np.mean([r['collab_weight'] * r['collab_score'] for r in results])


# ============================================================
# 50 SYNTHETIC PROFILES
#
# Design rationale:
#   - Covers all 4 streams proportionally to database distribution
#   - Covers all 6 RIASEC types for each stream where careers exist
#   - Includes edge cases: vague queries, interest/RIASEC mismatch,
#     cross-stream interest language
#   - Interest queries are realistic Class 12 student language
#     (no job titles mentioned — mirrors real usage)
# ============================================================

PROFILES = [

    # ── SCIENCE / Investigative (most common combination) ───
    {"id": "EVAL_S_I_01", "name": "Aditya",   "stream": "Science",
     "riasec": {"top2": ["I","R"]},
     "interest": "I love studying living organisms and how diseases spread in populations",
     "interest_raw": "I love biology and diseases"},

    {"id": "EVAL_S_I_02", "name": "Sneha",    "stream": "Science",
     "riasec": {"top2": ["I","C"]},
     "interest": "I enjoy solving complex mathematical problems and working with data patterns",
     "interest_raw": "I like maths and data"},

    {"id": "EVAL_S_I_03", "name": "Rohan",    "stream": "Science",
     "riasec": {"top2": ["I","R"]},
     "interest": "I find chemistry experiments fascinating especially understanding reactions at molecular level",
     "interest_raw": "I like chemistry"},

    {"id": "EVAL_S_I_04", "name": "Divya",    "stream": "Science",
     "riasec": {"top2": ["I","S"]},
     "interest": "I am curious about how the human brain works and what causes mental illness",
     "interest_raw": "I am curious about the brain"},

    {"id": "EVAL_S_I_05", "name": "Karthik",  "stream": "Science",
     "riasec": {"top2": ["I","R"]},
     "interest": "I want to study stars galaxies and the physics of the universe",
     "interest_raw": "I like stars and space"},

    # ── SCIENCE / Realistic ──────────────────────────────────
    {"id": "EVAL_S_R_01", "name": "Suresh",   "stream": "Science",
     "riasec": {"top2": ["R","I"]},
     "interest": "I enjoy building electronic circuits and working with microcontrollers and sensors",
     "interest_raw": "I like electronics"},

    {"id": "EVAL_S_R_02", "name": "Meena",    "stream": "Science",
     "riasec": {"top2": ["R","C"]},
     "interest": "I like working with machines understanding how engines and mechanical systems function",
     "interest_raw": "I like machines"},

    {"id": "EVAL_S_R_03", "name": "Pranav",   "stream": "Science",
     "riasec": {"top2": ["R","I"]},
     "interest": "I am interested in construction civil structures and how large infrastructure projects are designed",
     "interest_raw": "I like buildings and construction"},

    # ── SCIENCE / Social ─────────────────────────────────────
    {"id": "EVAL_S_S_01", "name": "Lakshmi",  "stream": "Science",
     "riasec": {"top2": ["S","I"]},
     "interest": "I want to help sick people recover and provide care to patients in hospitals",
     "interest_raw": "I want to help sick people"},

    {"id": "EVAL_S_S_02", "name": "Vijay",    "stream": "Science",
     "riasec": {"top2": ["S","A"]},
     "interest": "I enjoy teaching science concepts to younger students and explaining difficult topics clearly",
     "interest_raw": "I like teaching"},

    # ── SCIENCE / Conventional ───────────────────────────────
    {"id": "EVAL_S_C_01", "name": "Harini",   "stream": "Science",
     "riasec": {"top2": ["C","I"]},
     "interest": "I like organising and analysing large amounts of scientific data using computers",
     "interest_raw": "I like organising data"},

    # ── SCIENCE / Edge case: vague query ────────────────────
    {"id": "EVAL_S_VAGUE", "name": "Ravi",    "stream": "Science",
     "riasec": {"top2": ["I","R"]},
     "interest": "I like science",
     "interest_raw": "I like science"},

    # ── SCIENCE / Edge case: interest-RIASEC mismatch ───────
    # Student is Artistic but in Science — tests if system respects interest over RIASEC
    {"id": "EVAL_S_MISMATCH", "name": "Ananya", "stream": "Science",
     "riasec": {"top2": ["A","S"]},
     "interest": "I love designing graphics visualising scientific data and creating informative illustrations",
     "interest_raw": "I like drawing and science"},

    # ── COMMERCE / Enterprising ──────────────────────────────
    {"id": "EVAL_C_E_01", "name": "Rahul",    "stream": "Commerce",
     "riasec": {"top2": ["E","C"]},
     "interest": "I want to start my own company manage teams and grow a business from nothing",
     "interest_raw": "I want to start a business"},

    {"id": "EVAL_C_E_02", "name": "Pooja",    "stream": "Commerce",
     "riasec": {"top2": ["E","S"]},
     "interest": "I enjoy convincing people negotiating deals and understanding what customers need",
     "interest_raw": "I like sales"},

    {"id": "EVAL_C_E_03", "name": "Nikhil",   "stream": "Commerce",
     "riasec": {"top2": ["E","C"]},
     "interest": "I find financial markets exciting and want to understand stock trading and investment strategies",
     "interest_raw": "I like stocks and money"},

    {"id": "EVAL_C_E_04", "name": "Shreya",   "stream": "Commerce",
     "riasec": {"top2": ["E","S"]},
     "interest": "I want to work in marketing understand consumer behaviour and build brand strategies",
     "interest_raw": "I like marketing"},

    # ── COMMERCE / Conventional ──────────────────────────────
    {"id": "EVAL_C_C_01", "name": "Arjun",    "stream": "Commerce",
     "riasec": {"top2": ["C","E"]},
     "interest": "I enjoy working with numbers doing calculations auditing accounts and maintaining financial records",
     "interest_raw": "I like numbers and accounts"},

    {"id": "EVAL_C_C_02", "name": "Priya",    "stream": "Commerce",
     "riasec": {"top2": ["C","I"]},
     "interest": "I like analysing tax laws understanding company finances and working with spreadsheets",
     "interest_raw": "I like tax and finance"},

    {"id": "EVAL_C_C_03", "name": "Sanjay",   "stream": "Commerce",
     "riasec": {"top2": ["C","E"]},
     "interest": "I want to work in banking understand how loans work and help customers manage their money",
     "interest_raw": "I like banking"},

    # ── COMMERCE / Social ────────────────────────────────────
    {"id": "EVAL_C_S_01", "name": "Kavya",    "stream": "Commerce",
     "riasec": {"top2": ["S","E"]},
     "interest": "I enjoy helping people solve their problems especially in financial planning and insurance",
     "interest_raw": "I like helping people with money"},

    # ── COMMERCE / Edge case: vague query ───────────────────
    {"id": "EVAL_C_VAGUE", "name": "Deepak",  "stream": "Commerce",
     "riasec": {"top2": ["E","C"]},
     "interest": "I like commerce",
     "interest_raw": "I like commerce"},

    # ── COMMERCE / Edge case: cross-stream interest ─────────
    # Commerce student who writes like a Science student
    {"id": "EVAL_C_CROSS", "name": "Meghna",  "stream": "Commerce",
     "riasec": {"top2": ["I","C"]},
     "interest": "I enjoy analysing economic data understanding research methods and studying financial systems scientifically",
     "interest_raw": "I like research and economics"},

    # ── ARTS / Social ────────────────────────────────────────
    {"id": "EVAL_A_S_01", "name": "Nandini",  "stream": "Arts",
     "riasec": {"top2": ["S","A"]},
     "interest": "I want to work as a counsellor helping students deal with mental health and career confusion",
     "interest_raw": "I want to help students"},

    {"id": "EVAL_A_S_02", "name": "Arun",     "stream": "Arts",
     "riasec": {"top2": ["S","E"]},
     "interest": "I care about social justice poverty and want to work in community development NGOs",
     "interest_raw": "I like helping poor people"},

    {"id": "EVAL_A_S_03", "name": "Bhavna",   "stream": "Arts",
     "riasec": {"top2": ["S","I"]},
     "interest": "I enjoy understanding how society works studying human behaviour and social patterns",
     "interest_raw": "I like studying society"},

    # ── ARTS / Artistic ──────────────────────────────────────
    {"id": "EVAL_A_A_01", "name": "Chitra",   "stream": "Arts",
     "riasec": {"top2": ["A","S"]},
     "interest": "I love drawing painting creating visual designs and expressing emotions through art",
     "interest_raw": "I love drawing"},

    {"id": "EVAL_A_A_02", "name": "Farhan",   "stream": "Arts",
     "riasec": {"top2": ["A","E"]},
     "interest": "I am passionate about storytelling writing screenplays and creating films",
     "interest_raw": "I like movies and stories"},

    {"id": "EVAL_A_A_03", "name": "Geeta",    "stream": "Arts",
     "riasec": {"top2": ["A","S"]},
     "interest": "I enjoy dancing classical music and performing arts on stage in front of audiences",
     "interest_raw": "I like dancing and music"},

    {"id": "EVAL_A_A_04", "name": "Harsha",   "stream": "Arts",
     "riasec": {"top2": ["A","I"]},
     "interest": "I love photography capturing moments telling stories through images and editing photos",
     "interest_raw": "I like photography"},

    # ── ARTS / Enterprising ──────────────────────────────────
    {"id": "EVAL_A_E_01", "name": "Indira",   "stream": "Arts",
     "riasec": {"top2": ["E","S"]},
     "interest": "I want to go into politics public service and work to improve governance in India",
     "interest_raw": "I like politics"},

    # ── ARTS / Realistic ─────────────────────────────────────
    {"id": "EVAL_A_R_01", "name": "Jay",      "stream": "Arts",
     "riasec": {"top2": ["R","S"]},
     "interest": "I love playing football training hard and want to become a professional sportsperson",
     "interest_raw": "I like sports"},

    # ── ARTS / Edge case: vague query ───────────────────────
    {"id": "EVAL_A_VAGUE", "name": "Komal",   "stream": "Arts",
     "riasec": {"top2": ["S","A"]},
     "interest": "I like arts",
     "interest_raw": "I like arts"},

    # ── VOCATIONAL / Realistic ───────────────────────────────
    {"id": "EVAL_V_R_01", "name": "Lokesh",   "stream": "Vocational",
     "riasec": {"top2": ["R","C"]},
     "interest": "I enjoy repairing electrical systems fixing wiring and working with power tools",
     "interest_raw": "I like fixing electrical things"},

    {"id": "EVAL_V_R_02", "name": "Mohan",    "stream": "Vocational",
     "riasec": {"top2": ["R","I"]},
     "interest": "I like working with vehicles maintaining car engines and diagnosing mechanical faults",
     "interest_raw": "I like cars and engines"},

    {"id": "EVAL_V_R_03", "name": "Neeraj",   "stream": "Vocational",
     "riasec": {"top2": ["R","C"]},
     "interest": "I enjoy welding metal fabrication and hands-on construction and assembly work",
     "interest_raw": "I like welding and metal work"},

    {"id": "EVAL_V_R_04", "name": "Omkar",    "stream": "Vocational",
     "riasec": {"top2": ["R","E"]},
     "interest": "I want to work in agriculture farming crops understanding soil and irrigation systems",
     "interest_raw": "I like farming"},

    {"id": "EVAL_V_R_05", "name": "Pavan",    "stream": "Vocational",
     "riasec": {"top2": ["R","C"]},
     "interest": "I am interested in plumbing pipefitting and maintaining water and drainage systems",
     "interest_raw": "I like plumbing"},

    # ── VOCATIONAL / Conventional ────────────────────────────
    {"id": "EVAL_V_C_01", "name": "Qadir",    "stream": "Vocational",
     "riasec": {"top2": ["C","R"]},
     "interest": "I want to manage inventory control warehouse operations and logistics supply chains",
     "interest_raw": "I like warehouse work"},

    {"id": "EVAL_V_C_02", "name": "Ramesh",   "stream": "Vocational",
     "riasec": {"top2": ["C","E"]},
     "interest": "I enjoy managing records data entry and administrative office operations",
     "interest_raw": "I like office work"},

    # ── VOCATIONAL / Enterprising ────────────────────────────
    {"id": "EVAL_V_E_01", "name": "Savita",   "stream": "Vocational",
     "riasec": {"top2": ["E","R"]},
     "interest": "I want to supervise construction teams manage worksites and coordinate building projects",
     "interest_raw": "I like managing construction"},

    # ── VOCATIONAL / Social ──────────────────────────────────
    {"id": "EVAL_V_S_01", "name": "Tarun",    "stream": "Vocational",
     "riasec": {"top2": ["S","R"]},
     "interest": "I want to train people in technical skills work as an instructor in vocational institutes",
     "interest_raw": "I like teaching skills"},

    # ── VOCATIONAL / Edge case: vague query ─────────────────
    {"id": "EVAL_V_VAGUE", "name": "Uma",     "stream": "Vocational",
     "riasec": {"top2": ["R","C"]},
     "interest": "I like doing practical work with my hands",
     "interest_raw": "I like practical work"},

    # ── VOCATIONAL / Edge case: interest-RIASEC mismatch ────
    # Student surveys as Investigative but their interest is clearly hands-on
    {"id": "EVAL_V_MISMATCH", "name": "Vikas", "stream": "Vocational",
     "riasec": {"top2": ["I","R"]},
     "interest": "I enjoy outdoor physical work maintaining machines operating heavy equipment on construction sites",
     "interest_raw": "I like heavy machines outdoors"},

    # ── CROSS-STREAM edge cases ──────────────────────────────
    # Science student writing like they have Commerce interests
    {"id": "EVAL_CROSS_SC", "name": "Waqar",  "stream": "Science",
     "riasec": {"top2": ["E","I"]},
     "interest": "I want to use science knowledge to run a pharmaceutical business and manage health companies",
     "interest_raw": "I like science business"},

    # Arts student writing like they have Science interests
    {"id": "EVAL_CROSS_AS", "name": "Xena",   "stream": "Arts",
     "riasec": {"top2": ["I","A"]},
     "interest": "I am fascinated by human psychology research studying behaviour patterns through experiments",
     "interest_raw": "I like psychology research"},

    # Commerce student interested in creative work
    {"id": "EVAL_CROSS_CA", "name": "Yogesh", "stream": "Commerce",
     "riasec": {"top2": ["A","E"]},
     "interest": "I want to work in advertising creative campaigns brand design and visual communication",
     "interest_raw": "I like ads and design"},

    # Two profiles with same RIASEC but very different interests
    # (used to test personalization — should give different results)
    {"id": "EVAL_PERS_01", "name": "Zara",    "stream": "Science",
     "riasec": {"top2": ["I","R"]},
     "interest": "I am deeply interested in genomics DNA sequencing and understanding genetic diseases",
     "interest_raw": "I like genetics"},

    {"id": "EVAL_PERS_02", "name": "Zubin",   "stream": "Science",
     "riasec": {"top2": ["I","R"]},
     "interest": "I love coding writing algorithms building software and solving programming problems",
     "interest_raw": "I like coding"},

    # ── ARTS / Conventional (under-represented type) ────────
    {"id": "EVAL_A_C_01", "name": "Aisha",   "stream": "Arts",
     "riasec": {"top2": ["C","S"]},
     "interest": "I enjoy maintaining archives organising historical records and managing library collections",
     "interest_raw": "I like organising records"},
]

assert len(PROFILES) == N_PROFILES, f"Expected {N_PROFILES} profiles, got {len(PROFILES)}"
print(f"\n✅ {len(PROFILES)} synthetic profiles defined")


# ============================================================
# SIMULATE KEEP/DROP INTERACTIONS FOR STAGES 1-3
#
# Logic: for each profile, we recommend careers in Stage 0,
# then simulate Keep/Drop based on ground truth relevance.
# Keep = 5 (rating), Drop = 1 (rating).
# We generate enough signals to reach each stage threshold.
#
# Stage 1: 5 interactions per profile  (within 1-9 range)
# Stage 2: 15 interactions per profile (within 10-29 range)
# Stage 3: 35 interactions per profile (>= 30)
#
# New student IDs used (EVAL_ prefix) — not in seed data.
# ============================================================

def simulate_interactions(profile, n_interactions, seed_results):
    """
    Generate synthetic Keep/Drop interactions.
    Relevant careers (matching stream + RIASEC) get Keep (5).
    Others get Drop (1).
    We cycle through seed_results repeatedly if needed.
    """
    interactions = []
    pool = seed_results * (n_interactions // len(seed_results) + 2)
    for i in range(n_interactions):
        career = pool[i]
        rating = 5 if is_relevant(career, profile) else 1
        interactions.append({
            'student_id': profile['id'],
            'career_id':  career['career_id'],
            'rating':     rating,
            'weight':     2.0   # real signal weight
        })
    return interactions


# ============================================================
# CORE EVALUATION RUNNER
# ============================================================

def evaluate_stage(profiles, svd_model, n_real, stage_name,
                   stage0_results_map=None, mode="full"):
    """
    Run evaluation for one stage across all profiles.
    Returns a dict of aggregate metrics.
    """
    results_per_profile = {}
    all_careers_seen    = set()

    riasec_aligns   = []
    stream_rels     = []
    precisions      = []
    ndcgs           = []
    jaccard_dists   = []
    svd_contribs    = []

    for p in profiles:
        query  = p['interest']
        result = recommend(
            user_id             = p['id'],
            query_text          = query,
            student_riasec      = p['riasec'],
            n_real_interactions = n_real,
            preferred_stream    = p['stream'],
            svd_model           = svd_model,
            mode                = mode,
            top_k               = TOP_K
        )

        results_per_profile[p['id']] = result

        for r in result:
            all_careers_seen.add(r['career'])

        riasec_aligns.append(riasec_alignment(result, p))
        stream_rels.append(stream_relevance(result, p))
        precisions.append(precision_at_k(result, p))
        ndcgs.append(ndcg_at_k(result, p))
        svd_contribs.append(mean_svd_contribution(result))

        if stage0_results_map and p['id'] in stage0_results_map:
            jd = jaccard_distance(result, stage0_results_map[p['id']])
            jaccard_dists.append(jd)

    metrics = {
        'stage':              stage_name,
        'n_real':             n_real,
        'mode':               mode,
        'riasec_align':       round(np.mean(riasec_aligns) * 100, 1),
        'stream_relevance':   round(np.mean(stream_rels) * 100, 1),
        'precision_at_5':     round(np.mean(precisions) * 100, 1),
        'ndcg_at_5':          round(np.mean(ndcgs), 3),
        'unique_careers':     len(all_careers_seen),
        'catalog_coverage':   round(len(all_careers_seen) / len(df) * 100, 1),
        'mean_svd_contrib':   round(np.mean(svd_contribs), 4),
        'mean_jaccard_dist':  round(np.mean(jaccard_dists), 3) if jaccard_dists else None,
    }
    return metrics, results_per_profile


# ============================================================
# PERSONALIZATION CHECK
# Two profiles with same RIASEC but different interests.
# They should get different recommendations — proves
# semantic matching drives personalization, not just RIASEC.
# ============================================================

def personalization_check(svd_model, n_real=0):
    p1 = next(p for p in PROFILES if p['id'] == 'EVAL_PERS_01')
    p2 = next(p for p in PROFILES if p['id'] == 'EVAL_PERS_02')

    r1 = recommend(p1['id'], p1['interest'], p1['riasec'],
                   n_real, p1['stream'], svd_model)
    r2 = recommend(p2['id'], p2['interest'], p2['riasec'],
                   n_real, p2['stream'], svd_model)

    jd = jaccard_distance(r1, r2)
    t1 = [r['career'] for r in r1]
    t2 = [r['career'] for r in r2]
    overlap = set(t1) & set(t2)

    print(f"\n{'='*65}")
    print("PERSONALIZATION CHECK")
    print(f"  Zara  (genomics interest, I/R): {t1}")
    print(f"  Zubin (coding interest,  I/R): {t2}")
    print(f"  Overlap: {overlap}")
    print(f"  Jaccard distance: {jd:.3f}  (higher = more personalized)")
    if jd > 0.5:
        print("  ✅ Good personalization — same RIASEC, different recommendations")
    else:
        print("  ⚠️  Low personalization — similar results despite different interests")
    print(f"{'='*65}")
    return jd


# ============================================================
# MAIN EVALUATION FLOW
# ============================================================

print("\n" + "="*65)
print("STARTING FULL EVALUATION")
print("="*65)

all_metrics = []

# ── STAGE 0 ─────────────────────────────────────────────────
print("\n🔄 Stage 0 — Pure content (n_real=0)...")
stage0_metrics, stage0_results = evaluate_stage(
    PROFILES, svd_model=None, n_real=0,
    stage_name="Stage 0 (Cold Start)"
)
all_metrics.append(stage0_metrics)
print(f"  RIASEC Alignment : {stage0_metrics['riasec_align']}%")
print(f"  Stream Relevance : {stage0_metrics['stream_relevance']}%")
print(f"  Precision@5      : {stage0_metrics['precision_at_5']}%")
print(f"  NDCG@5           : {stage0_metrics['ndcg_at_5']}")
print(f"  Unique Careers   : {stage0_metrics['unique_careers']}")
print(f"  Catalog Coverage : {stage0_metrics['catalog_coverage']}%")

# ── PERSONALIZATION CHECK (Stage 0) ─────────────────────────
personalization_check(svd_model=None, n_real=0)

# ── SIMULATE STAGE 1 INTERACTIONS (5 per profile) ───────────
print("\n🔄 Simulating Stage 1 interactions (5 per profile)...")
stage1_interactions = []
for p in PROFILES:
    seed = stage0_results[p['id']]
    stage1_interactions.extend(simulate_interactions(p, 5, seed))

stage1_df  = pd.DataFrame(stage1_interactions)
svd_stage1 = LightSVD().fit(stage1_df)

stage1_metrics, stage1_results = evaluate_stage(
    PROFILES, svd_model=svd_stage1, n_real=5,
    stage_name="Stage 1 (Sparse)",
    stage0_results_map=stage0_results
)
all_metrics.append(stage1_metrics)
print(f"  RIASEC Alignment : {stage1_metrics['riasec_align']}%")
print(f"  Stream Relevance : {stage1_metrics['stream_relevance']}%")
print(f"  Precision@5      : {stage1_metrics['precision_at_5']}%")
print(f"  NDCG@5           : {stage1_metrics['ndcg_at_5']}")
print(f"  Jaccard from S0  : {stage1_metrics['mean_jaccard_dist']}")
print(f"  SVD Contribution : {stage1_metrics['mean_svd_contrib']}")

# ── SIMULATE STAGE 2 INTERACTIONS (15 per profile) ──────────
print("\n🔄 Simulating Stage 2 interactions (15 per profile)...")
stage2_interactions = []
for p in PROFILES:
    seed = stage0_results[p['id']]
    stage2_interactions.extend(simulate_interactions(p, 15, seed))

stage2_df  = pd.DataFrame(stage2_interactions)
svd_stage2 = LightSVD().fit(stage2_df)

stage2_metrics, stage2_results = evaluate_stage(
    PROFILES, svd_model=svd_stage2, n_real=15,
    stage_name="Stage 2 (Moderate)",
    stage0_results_map=stage0_results
)
all_metrics.append(stage2_metrics)
print(f"  RIASEC Alignment : {stage2_metrics['riasec_align']}%")
print(f"  Stream Relevance : {stage2_metrics['stream_relevance']}%")
print(f"  Precision@5      : {stage2_metrics['precision_at_5']}%")
print(f"  NDCG@5           : {stage2_metrics['ndcg_at_5']}")
print(f"  Jaccard from S0  : {stage2_metrics['mean_jaccard_dist']}")
print(f"  SVD Contribution : {stage2_metrics['mean_svd_contrib']}")

# ── SIMULATE STAGE 3 INTERACTIONS (35 per profile) ──────────
print("\n🔄 Simulating Stage 3 interactions (35 per profile)...")
stage3_interactions = []
for p in PROFILES:
    seed = stage0_results[p['id']]
    stage3_interactions.extend(simulate_interactions(p, 35, seed))

stage3_df  = pd.DataFrame(stage3_interactions)
svd_stage3 = LightSVD().fit(stage3_df)

stage3_metrics, stage3_results = evaluate_stage(
    PROFILES, svd_model=svd_stage3, n_real=35,
    stage_name="Stage 3 (Rich)",
    stage0_results_map=stage0_results
)
all_metrics.append(stage3_metrics)
print(f"  RIASEC Alignment : {stage3_metrics['riasec_align']}%")
print(f"  Stream Relevance : {stage3_metrics['stream_relevance']}%")
print(f"  Precision@5      : {stage3_metrics['precision_at_5']}%")
print(f"  NDCG@5           : {stage3_metrics['ndcg_at_5']}")
print(f"  Jaccard from S0  : {stage3_metrics['mean_jaccard_dist']}")
print(f"  SVD Contribution : {stage3_metrics['mean_svd_contrib']}")

# ── ABLATION STUDY (Stage 0 only) ───────────────────────────
print("\n" + "="*65)
print("ABLATION STUDY — Stage 0, 5 variants")
print("="*65)

ablation_modes = [
    ("full",      "Full System"),
    ("no_llm",    "No LLM Expansion (raw query)"),
    ("no_riasec", "No RIASEC Boost"),
    ("no_stream", "No Stream Boost/Penalty"),
    ("baseline_b","Baseline B (RIASEC Lookup Only)"),
]

ablation_metrics = []
for mode_key, mode_label in ablation_modes:
    print(f"\n  Running: {mode_label}...")

    # For no_llm mode: use interest_raw instead of expanded interest
    ablation_profiles = []
    for p in PROFILES:
        p_copy = dict(p)
        if mode_key == "no_llm":
            p_copy['interest'] = p['interest_raw']
        ablation_profiles.append(p_copy)

    m, _ = evaluate_stage(
        ablation_profiles, svd_model=None, n_real=0,
        stage_name=mode_label, mode=mode_key
    )
    ablation_metrics.append(m)
    print(f"    RIASEC Align: {m['riasec_align']}% | "
          f"Stream Rel: {m['stream_relevance']}% | "
          f"Precision@5: {m['precision_at_5']}% | "
          f"NDCG@5: {m['ndcg_at_5']}")

# ── SUMMARY TABLES ───────────────────────────────────────────
print("\n" + "="*65)
print("SUMMARY TABLE 1: Stage Progression")
print("="*65)
print(f"{'Stage':<25} {'RIASEC%':>8} {'Stream%':>8} {'P@5%':>8} "
      f"{'NDCG@5':>8} {'Coverage%':>10} {'Jaccard':>8} {'SVD_C':>7}")
print("-"*85)
for m in all_metrics:
    jd = f"{m['mean_jaccard_dist']:.3f}" if m['mean_jaccard_dist'] is not None else "  —  "
    print(f"{m['stage']:<25} {m['riasec_align']:>8} {m['stream_relevance']:>8} "
          f"{m['precision_at_5']:>8} {m['ndcg_at_5']:>8} "
          f"{m['catalog_coverage']:>10} {jd:>8} {m['mean_svd_contrib']:>7.4f}")

print("\n" + "="*65)
print("SUMMARY TABLE 2: Ablation Study (Stage 0)")
print("="*65)
print(f"{'Variant':<35} {'RIASEC%':>8} {'Stream%':>8} {'P@5%':>8} {'NDCG@5':>8}")
print("-"*70)
for m in ablation_metrics:
    print(f"{m['stage']:<35} {m['riasec_align']:>8} {m['stream_relevance']:>8} "
          f"{m['precision_at_5']:>8} {m['ndcg_at_5']:>8}")

# ── SAVE RESULTS ─────────────────────────────────────────────
all_rows = all_metrics + ablation_metrics
results_df = pd.DataFrame(all_rows)
results_df.to_csv("eval_results.csv", index=False)
print("\n✅ Results saved to eval_results.csv")

# ── PERSONALIZATION CHECK ACROSS STAGES ──────────────────────
print("\n" + "="*65)
print("PERSONALIZATION: Same RIASEC, Different Interests — Stage Comparison")
print("="*65)
for n_real, svd_m, label in [
    (0,  None,       "Stage 0"),
    (5,  svd_stage1, "Stage 1"),
    (15, svd_stage2, "Stage 2"),
    (35, svd_stage3, "Stage 3"),
]:
    p1 = next(p for p in PROFILES if p['id'] == 'EVAL_PERS_01')
    p2 = next(p for p in PROFILES if p['id'] == 'EVAL_PERS_02')
    r1 = recommend(p1['id'], p1['interest'], p1['riasec'], n_real, p1['stream'], svd_m)
    r2 = recommend(p2['id'], p2['interest'], p2['riasec'], n_real, p2['stream'], svd_m)
    jd = jaccard_distance(r1, r2)
    print(f"  {label}: Jaccard distance = {jd:.3f}  "
          f"({'more personalized' if jd > 0.5 else 'less personalized'})")

print("\n✅ Full evaluation complete.")
print("   Key outputs: eval_results.csv")
print("   Check summary tables above for paper reporting.")

✅ Pinecone API key loaded from Kaggle secrets

📂 Loading career database...
✅ Total careers: 905
stream
Vocational    303
Science       291
Commerce      180
Arts          131

🔄 Loading Sentence Transformer...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ SBERT loaded
🔄 Connecting to Pinecone...
✅ Pinecone connected

✅ 50 synthetic profiles defined

STARTING FULL EVALUATION

🔄 Stage 0 — Pure content (n_real=0)...
  RIASEC Alignment : 89.6%
  Stream Relevance : 87.2%
  Precision@5      : 86.0%
  NDCG@5           : 0.934
  Unique Careers   : 199
  Catalog Coverage : 22.0%

PERSONALIZATION CHECK
  Zara  (genomics interest, I/R): ['Genetic Counselors', 'Bioinformatics Scientists', 'Geneticists', 'Molecular and Cellular Biologists', 'Health Informatics Specialists']
  Zubin (coding interest,  I/R): ['Software Developers', 'Computer Programmers', 'Cartographers and Photogrammetrists', 'DevOps Engineer', 'Web Developers']
  Overlap: set()
  Jaccard distance: 1.000  (higher = more personalized)
  ✅ Good personalization — same RIASEC, different recommendations

🔄 Simulating Stage 1 interactions (5 per profile)...
  RIASEC Alignment : 86.0%
  Stream Relevance : 91.6%
  Precision@5      : 89.6%
  NDCG@5           : 0.979
  Jaccard from S0  : 0.2